# Step 5 · Who bears the loss: businesses, workers and residents

The steps so far all used visits: in counties with 72+ hours of outage in a week, visits to local places fell by about a quarter (Step 2); about three quarters of the activity residents lost at home turned up in other counties (Step 3); and the model missed the losses of large metros most (Step 4b). This step brings in other data to ask on whom the loss actually falls:

| Question | Approach |
| --- | --- |
| **Q1 Businesses and workers in the county, or the county's residents?** | Step 2c's model (event × week fixed effects, week-varying wind and rain controls) with other outcomes: small-business revenue and small businesses open (Womply), the number of employees paid, also split into wages below and above the median (Paychex), and residents' card spending (Affinity). All are put in the same unit as visits: log points from the county's own baseline |
| **Q2 Who could move their activity?** | Step 3's accounts ("lost at home" and "extra elsewhere" for each hit county-week), split by county median household income (lower and higher half) and by rural–urban status (USDA Rural-Urban Continuum Codes: metro and nonmetro), comparing the displacement share |
| **Q3 Whose losses did the model miss?** | Step 4's leave-one-storm-out predictions, comparing the residuals (actual − predicted) of two groups of counties among county-weeks with 24+ hours of outage. The rural–urban codes replace Step 4's crude "under 50,000 people is rural" |

**Why Q1 separates "in the county" from "residents"**: Womply counts businesses in the county where they are located, and Affinity counts cardholders in the county where they live (Paychex's documentation does not say which; it is treated here as employment in the county). If businesses in a county lost revenue while the county's residents did not spend less, the money followed the people, and the loss fell on local businesses and workers. This is Step 3's outflow of activity again, seen in two independent data sets.

**Why Q2 and Q3 compare within the same storm**: nearly all counties with 72+ hours of outage come from Ida and Laura, and the counties those two storms hit differ to begin with (Ida mostly the New Orleans and Baton Rouge metros, Laura more rural counties). Pooled, a difference between groups would mix in the difference between storms. Q2 therefore gives both groups the same mix of storms, and Q3 adds storm fixed effects.

**Data**: Step 1b's weekly panel (which already holds Womply and Affinity), Step 2a's wind and rain, Step 3's `step3_accounting.csv` and Step 4's `step4_predictions_loso.csv`; three files are downloaded: Opportunity Insights' county employment (Paychex, about 25 MB), the Census Bureau's SAIPE 2019 county median household income, and USDA's 2013 Rural-Urban Continuum Codes.

**Structure**: A data → B the decision rules, fixed in advance → C estimation functions → D Q1 → E Q2 → F Q3 → G two figures, checklist and conclusions. No Earth Engine is needed; it runs in a few minutes.

In [ ]:
# 1. Environment, folders, chart style
import os, sys, subprocess
from urllib.parse import quote
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.stats import norm
import requests
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display

try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive"
except ImportError:
    BASE = "."
OUT = f"{BASE}/outage_mobility"
NEED = {"step1b_events.csv": "Step 1b", "step1b_outage_event.csv": "Step 1b", "step1b_panel_weekly.csv.gz": "Step 1b",
        "oi_geoids_county.csv": "Step 1b", "step2a_hazard_weekly.csv": "Step 2a", "step3_accounting.csv": "Step 3",
        "step4_predictions_loso.csv": "Step 4"}
for f, step in NEED.items():
    if not os.path.exists(f"{OUT}/{f}"):
        raise FileNotFoundError(f"{OUT}/{f} is missing; run {step} first.")

def tidy(df, n=1):
    # round numeric columns only, to keep tables readable
    return df.round({c: n for c in df.select_dtypes("number").columns})

INK, MUTED, GRID, SURFACE, FAINT = "#0b0b0b", "#52514e", "#e6e5e1", "#fcfcfb", "#b9b8b2"
C_DARK, C_MID, C_LIGHT, C_GREY = "#104281", "#2a78d6", "#86b6ef", "#7a7974"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False, "axes.axisbelow": True,
    "text.color": INK, "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "legend.frameon": False, "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
})

In [ ]:
# 2. Parameters
HOLDOUT = ["uri2021"]                          # as before, left out
OUT_EDGES  = [0, 6, 24, 72, 120, np.inf]       # all bands and controls below are the same as in Step 2c
OUT_LEVELS = ["h000", "h006", "h024", "h072", "h120"]
GUST_EDGES = [-np.inf, 17.5, 25, 33, np.inf]
RAIN_EDGES = [-np.inf, 25, 75, 150, np.inf]
COLD_EDGES = [-np.inf, 5, 25, np.inf]
RAINCUM_EDGES = [-np.inf, 50, 150, 300, np.inf]
CUM_FROM = -1
TEST = "out4=h072p"                            # the coefficient Q1 looks at: 72+ h vs under 6 h
BASE_WEEKS = (-5, -2)                          # baseline weeks, as in Step 1b
BAD_WEEK_PCT = 1.0                             # baseline weeks with a mean outage share of 1% or more are not used (as in Step 1b)
MIN_BASE_WEEKS = 2
MIN_COUNTIES = 5                               # an outcome is judged only with at least this many counties at 72+ h
ALPHA = 0.05                                   # "significant": a wild cluster bootstrap p-value below this
MIN_CELL = 2                                   # Q2: a storm enters the comparison only with at least this many hit county-weeks in each group
AFFECTED_HOURS = 24                            # Q3: as in Step 4, only county-weeks with 24+ h of outage that week
METRO_MAX = 3                                  # Rural-Urban Continuum Codes 1–3 are metro, 4–9 nonmetro
B_BOOT = 999                                   # bootstrap replications
SEED = 2026
Z = norm.ppf(0.975)
OI_URL = "https://raw.githubusercontent.com/OpportunityInsights/EconomicTracker/main/data/"
SAIPE_URL = "https://www2.census.gov/programs-surveys/saipe/datasets/2019/2019-state-and-county/est19all.xls"
RUCC_URLS = ["https://www.ers.usda.gov/media/5769/2013-rural-urban-continuum-codes.xls?v=31457",   # 2013 edition (2010 census)
             "https://www.ers.usda.gov/media/5768/2023-rural-urban-continuum-codes.csv?v=30493"]   # fallback: 2023 edition; Connecticut uses planning regions, which do not match the old counties
PAGES = {"SAIPE": "https://www.census.gov/data/datasets/2019/demo/saipe/2019-state-and-county.html",
         "RUCC": "https://www.ers.usda.gov/data-products/rural-urban-continuum-codes"}
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"}

## A. Data

The same sample, bands and wind and rain controls as Step 2c (`MAIN`: 7 events without Uri; `flag_base`, `flag_prior` and `low_sample` left out). New variables:

- **Businesses and spending**: `revenue`, `merchants` (Womply) and `spend` (Affinity) in the Step 1b panel are "change since January 2020". One plus this value is an index; its log minus its mean over the county's clean baseline weeks (weeks −5 to −2 with a mean outage share under 1%), times 100, gives log points, as for visits
- **Employment**: Opportunity Insights' county Paychex data, the number of employees paid, with separate series for wages below and above the median. Weeks end on Friday and are matched to the Monday–Sunday week containing that Friday; converted as above
- **Two "same county-week" differences**: small-business revenue minus visits (`rev_minus_visits`), and low-wage minus high-wage employment (`emp_gap`). They exist only where the same county and week has both series, so differences between counties do not enter
- **Income**: the Census Bureau's SAIPE 2019 county median household income. Q2 and Q3 each split their counties into halves at the median of the counties being compared
- **Rural–urban**: USDA's 2013 Rural-Urban Continuum Codes, with 1–3 metro and 4–9 nonmetro. The 2013 edition has the same counties as the 2020–21 data; if it cannot be downloaded, the 2023 edition is used instead (Connecticut will be missing)

Downloaded files are saved to Drive, so a rerun does not download them again. If SAIPE or the rural–urban codes fail to download, follow the printed instructions to download them by hand into Drive and rerun the cell.

In [ ]:
# 3. Merge, bands, wind and rain controls (as in Step 2c)
EVENTS = pd.read_csv(f"{OUT}/step1b_events.csv", index_col=0)
for c in ["t0", "day0", "week0"]:
    EVENTS[c] = pd.to_datetime(EVENTS[c])
W  = pd.read_csv(f"{OUT}/step1b_panel_weekly.csv.gz", dtype={"fips": str})
L  = pd.read_csv(f"{OUT}/step1b_outage_event.csv", dtype={"fips": str})
HZ = pd.read_csv(f"{OUT}/step2a_hazard_weekly.csv", dtype={"fips": str})

HZ = HZ.sort_values(["event", "fips", "rel_week"])
after = HZ["rel_week"] >= CUM_FROM
grp = [HZ["event"], HZ["fips"]]
HZ["gust_cum"] = HZ["gust_mean_ms"].where(after).groupby(grp).cummax().fillna(HZ["gust_mean_ms"])
HZ["rain_cum"] = HZ["rain_mm"].where(after, 0).groupby(grp).cumsum().where(after, HZ["rain_mm"])

D = (W.merge(L[["event", "fips", "name", "group", "flag_base", "flag_prior"]], on=["event", "fips"])
      .merge(HZ[["event", "fips", "rel_week", "gust_mean_ms", "rain_mm", "cold_ppt_mm", "gust_cum", "rain_cum"]],
             on=["event", "fips", "rel_week"], how="left"))
D["week"]        = pd.to_datetime(D["week"]).astype("datetime64[ns]")
D["out_bin"]     = pd.cut(D["out_week_hours"], OUT_EDGES, right=False, labels=OUT_LEVELS).astype(str)
D["out4"]        = D["out_bin"].replace({"h072": "h072p", "h120": "h072p"})
D["gust_bin"]    = pd.cut(D["gust_mean_ms"], GUST_EDGES, labels=["g0", "g1", "g2", "g3"]).astype(str)
D["gustcum_bin"] = pd.cut(D["gust_cum"], GUST_EDGES, labels=["k0", "k1", "k2", "k3"]).astype(str)
D["rain_bin"]    = pd.cut(D["rain_mm"], RAIN_EDGES, labels=["r0", "r1", "r2", "r3"]).astype(str)
D["raincum_bin"] = pd.cut(D["rain_cum"], RAINCUM_EDGES, labels=["s0", "s1", "s2", "s3"]).astype(str)
D["cold_bin"]    = pd.cut(D["cold_ppt_mm"], COLD_EDGES, labels=["c0", "c1", "c2"]).astype(str)
D["ew"] = D["event"] + "_" + D["rel_week"].astype(str)                        # event × week
D["pw"] = ["pre" if r <= -2 else ("4+" if r >= 4 else str(r)) for r in D["rel_week"]]
D["gw"] = np.where(D["gustcum_bin"] == "k0", "base", D["gustcum_bin"] + "|" + D["pw"])
D["rw"] = np.where(D["raincum_bin"] == "s0", "base", D["raincum_bin"] + "|" + D["pw"])
NEEDED = ["out_week_hours", "gust_mean_ms", "rain_mm", "cold_ppt_mm", "gust_cum", "rain_cum"]
D["keep"] = ~D["flag_base"] & ~D["flag_prior"] & ~D["low_sample"] & D[NEEDED].notna().all(axis=1)
print(f"Panel: {len(D):,} county-weeks; {int(D['keep'].sum()):,} kept (Uri included)")

In [ ]:
# 4. Businesses, spending, employment: log points from the baseline
def fetch(url, path, min_bytes):
    # download once into Drive; skip if it is already there
    if os.path.exists(path) and os.path.getsize(path) >= min_bytes:
        return path
    r = requests.get(url, timeout=600, headers=HEADERS)
    r.raise_for_status()
    if len(r.content) < min_bytes:
        raise ValueError(f"file too small ({len(r.content):,} bytes), probably an error page")
    with open(path, "wb") as fh:
        fh.write(r.content)
    return path

EMP_COLS = ["emp", "emp_incbelowmed", "emp_incabovemed"]
try:
    path = fetch(OI_URL + quote("Employment - County - Weekly.csv"), f"{OUT}/oi_employment_county.csv", 1_000_000)
    emp = pd.read_csv(path, na_values=".", usecols=["year", "month", "day_endofweek", "countyfips"] + EMP_COLS)
    end = pd.to_datetime(emp[["year", "month", "day_endofweek"]].rename(columns={"day_endofweek": "day"}))
    emp["week"] = (end - pd.to_timedelta(end.dt.dayofweek, unit="D")).astype("datetime64[ns]")   # the Monday of the week containing the end date, aligned with GeoDS weeks
    emp["fips"] = emp["countyfips"].astype(int).astype(str).str.zfill(5)
    emp = emp.groupby(["fips", "week"], as_index=False)[EMP_COLS].mean()
    print("Paychex weeks end on weekday", sorted(int(x) for x in end.dt.dayofweek.unique()), "(0 = Monday)")
except Exception as err:
    emp = pd.DataFrame({"fips": pd.Series(dtype=str), "week": pd.Series(dtype="datetime64[ns]"), **{c: pd.Series(dtype=float) for c in EMP_COLS}})
    print("Downloading the Paychex employment data failed; the worker part of Q1 will show too few counties:", err)
D = D.merge(emp, on=["fips", "week"], how="left")

clean = D["rel_week"].between(*BASE_WEEKS) & ~(D["out_week_mean_pct"] >= BAD_WEEK_PCT)
by = [D["event"], D["fips"]]
def log_change(series):
    # 1 + "change since the base period" is an index; 100 × (log index − its mean over clean baseline weeks), the same log points as for visits
    lvl = 1 + pd.to_numeric(series, errors="coerce")
    lg = np.log(lvl.where(lvl > 0))
    b = lg.where(clean).groupby(by).transform("mean")
    n = lg.where(clean).groupby(by).transform("count")
    return ((lg - b) * 100).where(n >= MIN_BASE_WEEKS)

SERIES = {"revenue": "revenue_lp", "merchants": "merchants_lp", "spend": "spend_lp",
          "emp": "emp_lp", "emp_incbelowmed": "emp_lo_lp", "emp_incabovemed": "emp_hi_lp"}
for raw, new in SERIES.items():
    D[new] = log_change(D[raw])
D["rev_minus_visits"] = D["revenue_lp"] - D["visits_in_chg"]                # same county-week: small-business revenue minus visits
D["emp_gap"] = D["emp_lo_lp"] - D["emp_hi_lp"]                              # same county-week: low-wage minus high-wage employment
MAIN = D[D["keep"] & ~D["event"].isin(HOLDOUT)].copy()

LABELS = {"visits_in_chg": "Visits to places in the county (GeoDS)",
          "revenue_lp": "Small-business revenue (Womply)",
          "merchants_lp": "Small businesses open (Womply)",
          "emp_lp": "Employees paid, all wages (Paychex)",
          "emp_lo_lp": "Employees paid, below-median wages",
          "emp_hi_lp": "Employees paid, above-median wages",
          "visits_res_chg": "Residents' visits anywhere (GeoDS)",
          "spend_lp": "Residents' card spending (Affinity)",
          "rev_minus_visits": "Revenue minus visits",
          "emp_gap": "Below- minus above-median-wage employees"}
rows = []
for y, label in LABELS.items():
    d = MAIN[MAIN[y].notna()]
    h = d[d["out4"] == "h072p"]
    rows.append({"outcome": label, "county_weeks": len(d), "counties": d["fips"].nunique(),
                 "county_weeks_72plus": len(h), "counties_72plus": h["fips"].nunique()})
print(f"Main sample: {len(MAIN):,} county-weeks. Coverage of each outcome:")
pd.DataFrame(rows).set_index("outcome")

In [ ]:
# 5. County median household income (SAIPE 2019) and Rural-Urban Continuum Codes (USDA)
def raw_table(path):
    # read without a header (.xls needs xlrd; installed if missing); the header row is found afterwards
    if path.endswith(".csv"):
        return pd.read_csv(path, header=None, dtype=str, encoding="latin-1")
    try:
        return pd.read_excel(path, header=None, dtype=str)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xlrd"], check=True)
        return pd.read_excel(path, header=None, dtype=str)

def with_header(raw, words):
    # among the first 30 rows, the first containing all these words is the header
    for i in range(min(30, len(raw))):
        heads = [str(c).strip().lower() for c in raw.iloc[i]]
        if all(any(w in h for h in heads) for w in words):
            t = raw.iloc[i + 1:].reset_index(drop=True)
            t.columns = range(t.shape[1])
            return t, heads
    raise ValueError(f"no header row containing {words}")

def col(heads, word):
    return next(i for i, h in enumerate(heads) if word in h)

def county_values(fips, values):
    # five-digit county code → value; national and state rows (county code 000) and blanks are dropped
    f = pd.to_numeric(pd.Series(np.asarray(fips)), errors="coerce")
    v = pd.to_numeric(pd.Series(np.asarray(values)), errors="coerce")
    ok = f.notna() & v.notna() & (f % 1000 != 0)
    return pd.Series(v[ok].to_numpy(), index=f[ok].astype(int).astype(str).str.zfill(5))

def parse_saipe(path):
    t, heads = with_header(raw_table(path), ["state fips", "county fips", "median household income"])
    st, co = (pd.to_numeric(t[col(heads, w)], errors="coerce") for w in ["state fips", "county fips"])
    return county_values(st * 1000 + co, t[col(heads, "median household income")])

def parse_rucc(path):
    raw = raw_table(path)
    try:
        t, heads = with_header(raw, ["fips", "rucc"])
        return county_values(t[col(heads, "fips")], t[col(heads, "rucc")])
    except ValueError:
        t, heads = with_header(raw, ["fips", "attribute", "value"])          # the 2023 edition is a long table, with the variable named in the Attribute column
        t = t[t[col(heads, "attribute")].astype(str).str.upper().str.startswith("RUCC")]
        return county_values(t[col(heads, "fips")], t[col(heads, "value")])

def first_working(sources, parse, what):
    # try each source in turn and return the first that yields over 3,000 counties
    for url, path in sources:
        try:
            fetch(url, path, 20_000)
            s = parse(path)
            if len(s) < 3000:
                raise ValueError(f"only {len(s)} counties read")
            print(f"{what}: {os.path.basename(path)}, {len(s):,} counties")
            return s, os.path.basename(path)
        except Exception as err:
            print(f"{what}: {os.path.basename(path)} failed: {err}")
    print(f"Every source for {what} failed. Download it by hand from {PAGES[what]}, rename it {os.path.basename(sources[0][1])}, put it in {OUT} and rerun this cell")
    return None, None

INCOME, INCOME_SRC = first_working([(SAIPE_URL, f"{OUT}/step5_saipe2019.xls")], parse_saipe, "SAIPE")
RUCC, RUCC_SRC = first_working(list(zip(RUCC_URLS, [f"{OUT}/step5_rucc2013.xls", f"{OUT}/step5_rucc2023.csv"])), parse_rucc, "RUCC")
if RUCC_SRC is not None and "2023" in RUCC_SRC:
    print("Note: the 2023 rural-urban codes are in use; Connecticut's counties will be missing")

OI = pd.read_csv(f"{OUT}/oi_geoids_county.csv")
OI["fips"] = OI["countyfips"].astype(int).astype(str).str.zfill(5)
ATTR = pd.DataFrame(index=pd.Index(sorted(set(D["fips"])), name="fips"))
ATTR["name"] = pd.Series((OI["countyname"] + ", " + OI["stateabbrev"]).to_numpy(), index=OI["fips"]).reindex(ATTR.index)
ATTR["pop2019"] = OI.set_index("fips")["county_pop2019"].astype(float).reindex(ATTR.index)
ATTR["income"] = np.nan if INCOME is None else INCOME.reindex(ATTR.index)
ATTR["rucc"] = np.nan if RUCC is None else RUCC.reindex(ATTR.index)
ATTR["metro"] = (ATTR["rucc"] <= METRO_MAX).astype(float).where(ATTR["rucc"].notna())     # 1 = metro, 0 = nonmetro
ATTR.to_csv(f"{OUT}/step5_county_attributes.csv")

HITC = (MAIN[MAIN["out4"] == "h072p"].groupby("fips")
        .agg(events=("event", lambda x: ", ".join(sorted(set(x)))), weeks_72plus=("rel_week", "size"), max_hours=("out_week_hours", "max"))
        .join(ATTR[["name", "pop2019", "income", "rucc"]]))
HITC.to_csv(f"{OUT}/step5_hit_counties.csv")
print(f"Of the {len(HITC)} counties with 72+ h: {int(HITC['income'].notna().sum())} have income, {int(HITC['rucc'].notna().sum())} have a rural-urban code")
tidy(HITC.sort_values("income"), 0)

## B. The decision rules, fixed in advance

Written down before looking at the results, with one caveat: Step 2b already showed Womply and Affinity in percentage points with cruder wind and rain controls (small-business revenue and businesses open fell, residents' card spending rose), so the rules for Q1 (a) and (c) are not fully in advance; the results for workers, Q1 (b)(d), Q2 and Q3 have not been seen.

"Significant" always means a wild cluster bootstrap p-value < 0.05: only a few to a few dozen counties had 72+ hours of outage, and the bootstrap is more reliable than county-clustered intervals there. An outcome is judged only if at least 5 counties had 72+ hours of outage in it; otherwise the result is "too few counties to judge".

**Q1 On whom the loss falls** (72+ h vs under 6 h, HxW model)

- (a) **Local small businesses lost**: the coefficients for small-business revenue and for small businesses open are both below 0 and significant
- (b) **Low-wage workers**: low-wage minus high-wage employment in the same county-week; below 0 and significant → "low-wage employment fell more"; above 0 and significant → "high-wage employment fell more"; otherwise → "no clear difference"
- (c) **Residents' spending**: above 0 and significant → "residents spent more"; below 0 and significant → "residents also spent less"; otherwise → "no clear change"
- (d) **Do visits stand in for revenue**: small-business revenue minus visits in the same county-week; below 0 and significant → "revenue fell more than visits"; above 0 and significant → "revenue fell less than visits"; otherwise → "about as much as visits"

**Q2 Who could move their activity** (difference in the storm-adjusted displacement share, 95% interval from 999 resamples of counties)

- Income: higher half minus lower half; interval entirely above 0 → "higher-income counties moved more of their activity elsewhere"; entirely below 0 → "lower-income counties moved more of their activity elsewhere"; otherwise → "no clear difference"
- Rural–urban: metro minus nonmetro, by the same rule

**Q3 Whose losses the model missed** (difference in residuals within the same storm: storm fixed effects, clustered by county)

- Rural–urban: metro vs nonmetro; below 0 and significant → "the model underestimates metro losses more"; above 0 and significant → "the model underestimates nonmetro losses more"; otherwise → "no clear difference"
- Income: lower half vs higher half, by the same rule (below 0 → lower-income counties underestimated more)

Q2 and Q3 rest on a few dozen hit county-weeks, so their power is limited: "no clear difference" does not mean "no difference", and the conclusions give the intervals alongside.

## C. Estimation functions

Exactly as in Step 2c: fixed effects are subtracted before least squares, standard errors are clustered by county (CR1), and the wild cluster bootstrap uses Webb weights with the null imposed. `ledger()` runs the HxW model for one outcome and takes out the 72+ h coefficient.

In [ ]:
# 6. Fixed-effects least squares, clustered standard errors, wild cluster bootstrap (as in Step 2c)
OUT_X = [("out4", "h000")]
CONTROLS = [("gust_bin", "g0"), ("rain_bin", "r0"), ("cold_bin", "c0"), ("gw", "base"), ("rw", "base")]   # Step 2c's HxW controls
WEBB = np.array([-np.sqrt(1.5), -1.0, -np.sqrt(0.5), np.sqrt(0.5), 1.0, np.sqrt(1.5)])

def dummies(d, specs):
    # banded variables → 0/1 columns (reference band dropped), named "variable=band"
    parts = [pd.DataFrame(index=d.index)]
    for c, ref in specs:
        dm = pd.get_dummies(d[c].astype(str), prefix=c, prefix_sep="=", dtype=float)
        parts.append(dm.drop(columns=[f"{c}={ref}"], errors="ignore"))
    return pd.concat(parts, axis=1)

def demean(M, codes, tol=1e-10, max_iter=5000):
    # remove fixed effects: subtract group means for each set in turn until convergence (one pass is enough for a single set)
    M = np.array(M, dtype=float)
    groups = [(c, sp.csr_matrix((np.ones(len(c)), (c, np.arange(len(c))))), np.bincount(c)) for c in codes]
    scale = max(1.0, float(np.abs(M).max()))
    for _ in range(max_iter):
        change = 0.0
        for c, S, n in groups:
            means = (S @ M) / n[:, None]
            M -= means[c]
            change = max(change, float(np.abs(means).max()))
        if len(groups) <= 1 or change < tol * scale:
            break
    return M

def drop_singletons(d, fe):
    # observations alone in a fixed-effect group add nothing; drop them repeatedly until none are left
    while True:
        lone = np.zeros(len(d), bool)
        for f in fe:
            lone |= (d.groupby(f)[f].transform("size") < 2).to_numpy()
        if not lone.any():
            return d
        d = d[~lone]

def fe_ols(data, y, xspec, fe=("ew",), test=TEST, boot=0, seed=SEED):
    # fixed-effects least squares; county-clustered standard errors (CR1); optional wild cluster bootstrap (testing that `test` = 0)
    d = drop_singletons(data.dropna(subset=[y]), list(fe))
    X = dummies(d, xspec)
    X = X.loc[:, X.sum() > 0]
    codes = [pd.factorize(d[f])[0] for f in fe]
    Zm = demean(np.column_stack([d[y].to_numpy(float), X.to_numpy()]), codes)
    yv, Xm = Zm[:, 0], Zm[:, 1:]
    live = np.linalg.norm(Xm, axis=0) > 1e-7 * np.sqrt(len(yv))               # drop columns fully absorbed by the fixed effects
    names, Xm = list(X.columns[live]), Xm[:, live]
    N = len(yv)
    XtX_inv = np.linalg.pinv(Xm.T @ Xm)
    A = XtX_inv @ Xm.T
    b = A @ yv
    u = yv - Xm @ b
    g = pd.factorize(d["fips"])[0]
    G = int(g.max()) + 1
    C = sp.csr_matrix((np.ones(N), (g, np.arange(N))), shape=(G, N))
    nested = [f for f in fe if d.groupby(f)["fips"].nunique().max() == 1]   # fixed effects nested within counties are not counted in the degrees of freedom
    k_fe = sum(d[f].nunique() for f in fe if f not in nested) - max(0, len(fe) - 1)
    K = int(np.linalg.matrix_rank(Xm)) + k_fe
    corr = G / (G - 1) * (N - 1) / (N - K)
    meat = C @ (Xm * u[:, None])
    se = np.sqrt(np.clip(np.diag(corr * XtX_inv @ (meat.T @ meat) @ XtX_inv), 0, None))
    table = pd.DataFrame({"coef": b, "se": se, "lo": b - Z * se, "hi": b + Z * se}, index=names)
    c_, lev = test.split("=")
    treated = d[c_].astype(str) == lev
    out = {"table": table, "n": N, "clusters": G, "p_boot": np.nan,
           "n_test": int(treated.sum()), "counties_test": int(d.loc[treated, "fips"].nunique())}
    if boot and test in names:
        k = names.index(test)
        Xr = np.delete(Xm, k, axis=1)
        fit_r = Xr @ np.linalg.lstsq(Xr, yv, rcond=None)[0]                   # the model with the null (coefficient = 0) imposed
        u_r = yv - fit_r
        t_obs = b[k] / se[k]
        rng = np.random.default_rng(seed)
        t_star = []
        for start in range(0, boot, 100):
            v = rng.choice(WEBB, size=(G, min(100, boot - start)))           # one weight per county
            Ys = fit_r[:, None] + u_r[:, None] * v[g]
            Bs = A @ Ys
            s = C @ (A[k][:, None] * (Ys - Xm @ Bs))
            t_star.append(Bs[k] / np.sqrt(corr * (np.asarray(s) ** 2).sum(axis=0)))
        out["p_boot"] = float(np.mean(np.abs(np.concatenate(t_star)) >= abs(t_obs)))
    return out

LEDGER, BANDS = [], []
def ledger(y):
    # one outcome: the HxW model, keeping the 72+ h coefficient
    d = MAIN.dropna(subset=[y])
    hit = d["out4"] == "h072p"
    row = {"outcome": y, "label": LABELS[y], "coef": np.nan, "se": np.nan, "lo": np.nan, "hi": np.nan, "p_boot": np.nan,
           "n_obs": len(d), "n_72plus": int(hit.sum()), "counties_72plus": int(d.loc[hit, "fips"].nunique())}
    if row["counties_72plus"] >= 2:
        r = fe_ols(d, y, OUT_X + CONTROLS, boot=B_BOOT)
        t = r["table"]
        if TEST in t.index:
            row.update(t.loc[TEST, ["coef", "se", "lo", "hi"]].to_dict())
            row.update(p_boot=r["p_boot"], n_obs=r["n"], n_72plus=r["n_test"], counties_72plus=r["counties_test"])
        BANDS.append(t[t.index.str.startswith("out4=")].assign(outcome=y))
    LEDGER.append(row)
    print(f"  {LABELS[y]:<42} {row['coef']:+7.1f} [{row['lo']:+6.1f}, {row['hi']:+6.1f}]  p = {row['p_boot']:.3f}  "
          f"{row['n_72plus']} county-weeks / {row['counties_72plus']} counties")
    return row

def judged(r):
    return r is not None and pd.notna(r["coef"]) and r["counties_72plus"] >= MIN_COUNTIES

def three_way(r, neg, pos, none):
    # per the decision rules: significantly negative → neg, significantly positive → pos, otherwise → none
    if not judged(r):
        return "too few counties to judge"
    return (neg if r["coef"] < 0 else pos) if r["p_boot"] < ALPHA else none

## D. Q1: on whom the loss falls

Each outcome gets Step 2c's HxW model (event × week fixed effects, outage bands, this week's wind and rain and week-varying cumulative wind and rain), so every coefficient means the same thing: **within the same event and week, how many log points lower (or higher) a county with 72+ hours of outage was than a county with under 6 hours**. The first row (visits) is Step 2c's main result, and the checklist confirms the two match.

Coverage differs a lot between outcomes (see the table in part A): Womply covers only larger counties, so the small-business results come from a handful of counties and should be read with the county counts.

In [ ]:
# 7. Q1: the HxW model for each outcome
for y in LABELS:
    ledger(y)
LED = pd.DataFrame(LEDGER).set_index("outcome")
LED.to_csv(f"{OUT}/step5_ledger.csv")
if BANDS:
    pd.concat(BANDS).rename_axis("term").to_csv(f"{OUT}/step5_ledger_bands.csv")

R = {y: LED.loc[y] for y in LED.index}
rev, mer = R["revenue_lp"], R["merchants_lp"]
Q1A = None if not (judged(rev) and judged(mer)) else bool(rev["coef"] < 0 and mer["coef"] < 0 and rev["p_boot"] < ALPHA and mer["p_boot"] < ALPHA)
Q1B = three_way(R["emp_gap"], "low-wage employment fell more", "high-wage employment fell more", "no clear difference")
Q1C = three_way(R["spend_lp"], "residents also spent less", "residents spent more", "no clear change")
Q1D = three_way(R["rev_minus_visits"], "revenue fell more than visits", "revenue fell less than visits", "about as much as visits")
print()
print("Q1 (a) local small businesses lost:", "too few counties to judge" if Q1A is None else Q1A)
print("Q1 (b) low-wage workers:", Q1B)
print("Q1 (c) residents' spending:", Q1C)
print("Q1 (d) visits and revenue:", Q1D)

In [ ]:
# 8. Figure 1: on whom the loss falls
PANELS = [[("In the county", ["visits_in_chg", "revenue_lp", "merchants_lp", "emp_lp", "emp_lo_lp", "emp_hi_lp"], C_DARK),
           ("The county's residents, wherever they are", ["visits_res_chg", "spend_lp"], C_MID)],
          [("Same county-weeks", ["rev_minus_visits", "emp_gap"], INK)]]

def coef_rows(ax, groups):
    # each group gets a header row, then its coefficients; hollow means too few counties
    pos, ticks, labs, heads = 0.0, [], [], []
    for head, ys, color in groups:
        heads.append((pos, head))
        pos -= 1
        for y in ys:
            r = LED.loc[y]
            if pd.notna(r["coef"]):
                hollow = r["counties_72plus"] < MIN_COUNTIES
                ax.errorbar(r["coef"], pos, xerr=[[r["coef"] - r["lo"]], [r["hi"] - r["coef"]]], fmt="o", color=color,
                            mfc=SURFACE if hollow else color, mew=1.6, ms=6, elinewidth=1.6, capsize=0)
                ax.annotate(f"{r['coef']:+.0f}", (r["coef"], pos), xytext=(0, 6), textcoords="offset points", ha="center", va="bottom", fontsize=8.5, color=INK)
                note = f"{int(r['counties_72plus'])} counties, bootstrap p = {r['p_boot']:.3f}"
            else:
                note = "too few counties"
            ticks.append(pos)
            labs.append(f"{r['label']}\n{note}")
            pos -= 1
        pos -= 0.4
    ax.set_yticks(ticks)
    ax.set_yticklabels(labs, fontsize=8.5)
    for yh, head in heads:
        ax.text(0.0, yh, head, transform=ax.get_yaxis_transform(), ha="left", va="center", fontsize=9, fontweight="bold", color=INK)
    ax.set_ylim(pos + 0.6, 0.7)
    ax.axvline(0, color=FAINT, lw=1)
    ax.grid(axis="y", visible=False)

fig, axes = plt.subplots(1, 2, figsize=(13.5, 6.2), gridspec_kw={"width_ratios": [1.5, 1]})
for ax, groups in zip(axes, PANELS):
    coef_rows(ax, groups)
axes[1].set_ylim(axes[0].get_ylim())                                        # rows on the right line up with the left, starting from the top
axes[0].set_xlabel("Log points vs counties with under 6 h of outage, same storm and week (95% interval)")
axes[0].set_title("72+ h of outage in the week")
axes[1].set_xlabel("Difference in log points (95% interval)")
axes[1].set_title("Paired differences")
fig.suptitle("On whom the loss falls: county-weeks with 72+ h of outage, 7 storms", x=0.01, ha="left", fontweight="bold", fontsize=12)
fig.tight_layout()
fig.text(0.01, -0.01, f"Hollow: fewer than {MIN_COUNTIES} counties with 72+ h. Womply and Paychex count businesses and payrolls in the county; "
         "Affinity counts spending by the county's residents, wherever it happens.", ha="left", va="top", color=MUTED, fontsize=8.5)
fig.savefig(f"{OUT}/step5_fig1_ledger.png"); plt.show()

## E. Q2: who could move their activity

Step 3 already split each hit county-week into "lost at home" and "extra elsewhere" (`step3_accounting.csv`), with displacement share = extra elsewhere ÷ lost at home. Here the counties are split into two groups by income and by rural–urban status, and the share is computed for each.

**Storm-adjusted**: the two groups have different mixes of storms (most metro hit counties come from Ida, for example), so plain sums would count differences between storms as differences between groups. Within each storm, each group's mean "lost at home" and "extra elsewhere" per county-week are therefore computed, weighted by the storm's county-weeks in both groups together and summed, and the two sums divided. Both groups then have the same mix of storms. Only storms with at least 2 hit county-weeks in each group are used; the plain sums are reported alongside, for reference.

Intervals come from 999 resamples of counties (all weeks of a county drawn together).

In [ ]:
# 9. Q2: displacement share by income and by rural-urban status
ACC = pd.read_csv(f"{OUT}/step3_accounting.csv", dtype={"fips": str}).join(ATTR[["income", "metro"]], on="fips")
INC_CUT2 = ACC.drop_duplicates("fips")["income"].median()
ACC["higher_income"] = (ACC["income"] >= INC_CUT2).astype(float).where(ACC["income"].notna())
print(f"{len(ACC)} hit county-weeks in {ACC['fips'].nunique()} counties; income cut-off {INC_CUT2:,.0f} dollars (the median of these counties)")

def moved(a, col_, boot=B_BOOT, seed=SEED):
    # displacement share of two groups (col_ = 0 / 1): plain sums and storm-adjusted; resampling counties
    a = a.dropna(subset=[col_])
    if a[col_].nunique() < 2:
        return None
    keys = ["fips", "event", col_]
    T = a.groupby(keys)[["lost", "extra", "home_cf"]].sum().join(a.groupby(keys).size().rename("n")).reset_index()
    ci = pd.factorize(T["fips"])[0]
    G = int(ci.max()) + 1
    storms = list(dict.fromkeys(T["event"]))
    k = T["event"].map({e: i for i, e in enumerate(storms)}).to_numpy() * 2 + T[col_].astype(int).to_numpy()   # cell: storm × group

    def sums(w):
        return {v: np.bincount(k, weights=w[ci] * T[v].to_numpy(float), minlength=2 * len(storms)).reshape(-1, 2)
                for v in ["lost", "extra", "home_cf", "n"]}

    def stats(S):
        with np.errstate(divide="ignore", invalid="ignore"):                # a resample may draw no county of a group
            pooled = S["extra"].sum(axis=0) / S["lost"].sum(axis=0)
            use = (S["n"] >= MIN_CELL).all(axis=1)
            if not use.any():
                return np.r_[pooled, np.nan, np.nan, 0.0]
            n_e = S["n"][use].sum(axis=1, keepdims=True)                     # county-weeks of both groups together in the storm
            adj = (n_e * S["extra"][use] / S["n"][use]).sum(axis=0) / (n_e * S["lost"][use] / S["n"][use]).sum(axis=0)
        return np.r_[pooled, adj, float(use.sum())]

    S0 = sums(np.ones(G))
    point = stats(S0)
    rng = np.random.default_rng(seed)
    draws = np.array([stats(sums(np.bincount(rng.integers(0, G, G), minlength=G).astype(float))) for _ in range(boot)])
    pct = lambda x, q: float(np.percentile(x[np.isfinite(x)], q)) if np.isfinite(x).any() else np.nan
    adj_d, pool_d = draws[:, 3] - draws[:, 2], draws[:, 1] - draws[:, 0]
    by_storm = pd.DataFrame({"storm": np.repeat(storms, 2), "group": np.tile([0, 1], len(storms)),
                             "county_weeks": S0["n"].ravel(), "share": (S0["extra"] / S0["lost"]).ravel(),
                             "used": np.repeat((S0["n"] >= MIN_CELL).all(axis=1), 2)})
    return {"pooled": point[:2], "adj": point[2:4], "storms": int(point[4]),
            "adj_lo": [pct(draws[:, 2 + j], 2.5) for j in (0, 1)], "adj_hi": [pct(draws[:, 2 + j], 97.5) for j in (0, 1)],
            "diff": point[3] - point[2], "diff_lo": pct(adj_d, 2.5), "diff_hi": pct(adj_d, 97.5),
            "pooled_diff": point[1] - point[0], "pooled_lo": pct(pool_d, 2.5), "pooled_hi": pct(pool_d, 97.5),
            "county_weeks": S0["n"].sum(axis=0), "counties": [a.loc[a[col_] == j, "fips"].nunique() for j in (0, 1)],
            "home_pct": -S0["lost"].sum(axis=0) / S0["home_cf"].sum(axis=0) * 100,
            "else_pct": S0["extra"].sum(axis=0) / S0["home_cf"].sum(axis=0) * 100, "by_storm": by_storm}

SPLITS2 = {"income": ("higher_income", ["lower-income half", "higher-income half"]),
           "metro": ("metro", ["nonmetro", "metro"])}
Q2, rows, storm_rows = {}, [], []
for split, (c, names) in SPLITS2.items():
    r = moved(ACC, c)
    Q2[split] = r
    if r is None:
        continue
    r["names"] = names
    for j in (0, 1):
        rows.append({"split": split, "group": names[j], "county_weeks": r["county_weeks"][j], "counties": r["counties"][j],
                     "home_change_pct": r["home_pct"][j], "elsewhere_change_pct": r["else_pct"][j], "share_pooled": r["pooled"][j],
                     "share_adjusted": r["adj"][j], "adjusted_lo": r["adj_lo"][j], "adjusted_hi": r["adj_hi"][j],
                     "diff_adjusted": r["diff"], "diff_lo": r["diff_lo"], "diff_hi": r["diff_hi"],
                     "diff_pooled": r["pooled_diff"], "pooled_lo": r["pooled_lo"], "pooled_hi": r["pooled_hi"], "storms_used": r["storms"]})
    storm_rows.append(r["by_storm"].assign(split=split, group=lambda d, nm=names: d["group"].map(dict(enumerate(nm)))))
MOVED = pd.DataFrame(rows)
MOVED.to_csv(f"{OUT}/step5_who_moved.csv", index=False)
if storm_rows:
    pd.concat(storm_rows, ignore_index=True).to_csv(f"{OUT}/step5_who_moved_by_storm.csv", index=False)

Q2_TEXT = {"income": ("higher-income counties moved more of their activity elsewhere",
                      "lower-income counties moved more of their activity elsewhere"),
           "metro": ("metro counties moved more of their activity elsewhere",
                     "nonmetro counties moved more of their activity elsewhere")}
Q2_VERDICT = {}
for split, r in Q2.items():
    if r is None or r["storms"] == 0 or not np.isfinite(r["diff_lo"]):
        Q2_VERDICT[split] = "pending (no storm with both groups)"
    elif r["diff_lo"] > 0:
        Q2_VERDICT[split] = Q2_TEXT[split][0]
    elif r["diff_hi"] < 0:
        Q2_VERDICT[split] = Q2_TEXT[split][1]
    else:
        Q2_VERDICT[split] = "no clear difference"
    if r is not None:
        print(f"Q2 {split}: {r['names'][1]} − {r['names'][0]} = {r['diff']:+.2f} [{r['diff_lo']:+.2f}, {r['diff_hi']:+.2f}] "
              f"(storm-adjusted, {r['storms']} storms); plain sums {r['pooled_diff']:+.2f} → {Q2_VERDICT[split]}")
tidy(MOVED.set_index(["split", "group"])[["county_weeks", "counties", "home_change_pct", "elsewhere_change_pct", "share_pooled",
                                          "share_adjusted", "adjusted_lo", "adjusted_hi"]], 2) if len(MOVED) else None

## F. Q3: whose losses the model missed

Step 4's leave-one-storm-out predictions from the "network" model (`step4_predictions_loso.csv`), using only county-weeks in the first three weeks after the storm with 24+ hours of outage that week, as in Step 4's P4. Residual = actual − predicted; below 0 means visits fell more than predicted, i.e. the model **underestimated the loss**.

The residual is regressed on the group with storm fixed effects, so the coefficient is the difference between the two groups within the same storm, and storm-level errors such as "Ida as a whole was underestimated" are removed. Step 4's definition (under 50,000 people) is redone too; without storm fixed effects it should reproduce Step 4's number exactly.

In [ ]:
# 10. Q3: difference in residuals between two groups of counties within the same storm
P = pd.read_csv(f"{OUT}/step4_predictions_loso.csv", dtype={"fips": str})
A3 = P[P["is_eval"].astype(str).eq("True") & (P["out_week_hours"] >= AFFECTED_HOURS)].copy()
A3["res"] = A3["y"] - A3["p_net"]
A3 = A3.join(ATTR[["income", "metro"]], on="fips")
INC_CUT3 = A3.drop_duplicates("fips")["income"].median()
A3["metro_g"] = A3["metro"].map({1.0: "metro", 0.0: "nonmetro"})
A3["income_g"] = pd.Series(np.where(A3["income"] < INC_CUT3, "lower", "higher"), index=A3.index).where(A3["income"].notna())
A3["size_g"] = np.where(A3["rural"].astype(str).eq("True"), "small", "other")       # Step 4's definition: under 50,000 people
print(f"{len(A3)} county-weeks with 24+ h in {A3['fips'].nunique()} counties; income cut-off {INC_CUT3:,.0f} dollars")

SPLITS3 = [("metro_g", "nonmetro", "metro", "Metro vs nonmetro (RUCC)"),
           ("income_g", "higher", "lower", "Lower- vs higher-income half"),
           ("size_g", "other", "small", "Under 50k people vs other (Step 4)")]
rows, means = [], []
for c, ref, lev, label in SPLITS3:
    d = A3.dropna(subset=[c])
    row = {"split": c, "label": label, "group": lev, "reference": ref, "coef": np.nan, "se": np.nan, "lo": np.nan, "hi": np.nan, "p_boot": np.nan,
           "raw_gap": np.nan, "county_weeks": len(d), "counties_group": d.loc[d[c] == lev, "fips"].nunique(),
           "counties_reference": d.loc[d[c] == ref, "fips"].nunique()}
    if d[c].nunique() == 2:
        r = fe_ols(d, "res", [(c, ref)], fe=("event",), test=f"{c}={lev}", boot=B_BOOT)
        if f"{c}={lev}" in r["table"].index:
            row.update(r["table"].loc[f"{c}={lev}", ["coef", "se", "lo", "hi"]].to_dict())
            row["p_boot"] = r["p_boot"]
        row["raw_gap"] = d.loc[d[c] == lev, "res"].mean() - d.loc[d[c] == ref, "res"].mean()
        means.append(d.groupby(c).agg(county_weeks=("res", "size"), counties=("fips", "nunique"), mean_actual=("y", "mean"),
                                      mean_predicted=("p_net", "mean"), mean_residual=("res", "mean")).assign(split=label))
    rows.append(row)
MISS = pd.DataFrame(rows).set_index("split")
MISS.to_csv(f"{OUT}/step5_model_misses.csv")

Q3_TEXT = {"metro_g": ("the model underestimates metro losses more", "the model underestimates nonmetro losses more"),
           "income_g": ("the model underestimates lower-income losses more", "the model underestimates higher-income losses more")}
Q3_VERDICT = {}
for c, (neg, pos) in Q3_TEXT.items():
    r = MISS.loc[c]
    if pd.isna(r["coef"]) or min(r["counties_group"], r["counties_reference"]) < MIN_COUNTIES:
        Q3_VERDICT[c] = "too few counties to judge"
    else:
        Q3_VERDICT[c] = (neg if r["coef"] < 0 else pos) if r["p_boot"] < ALPHA else "no clear difference"
    print(f"Q3 {r['label']}: {r['coef']:+.1f} [{r['lo']:+.1f}, {r['hi']:+.1f}], p = {r['p_boot']:.3f} "
          f"(within storm; pooled {r['raw_gap']:+.1f}) → {Q3_VERDICT[c]}")
tidy(pd.concat(means).reset_index().rename(columns={"index": "group"}).set_index(["split", "group"]), 1) if means else None

## G. Figures, checklist and conclusions

- **Figure 1** (cell 8 above): the 72+ h coefficient for each outcome. Left, places, businesses and workers in the county, and the county's residents; right, the two "same county-week" differences
- **Figure 2**: left, Q2, the storm-adjusted displacement share of the two groups (the tick marks the plain sums); right, Q3, the difference in residuals between the two groups within the same storm

In [ ]:
# 11. Figure 2: who could move, whose losses the model missed
fig, (ax, bx) = plt.subplots(1, 2, figsize=(13.5, 4.6), gridspec_kw={"width_ratios": [1.15, 1]})
ypos, ticks, labs = 0.0, [], []
for split, title in [("income", "By county income"), ("metro", "By rural-urban status")]:
    r = Q2.get(split)
    if r is None:
        continue
    ax.text(0.0, ypos, f"{title}: difference {r['diff']:+.2f} [{r['diff_lo']:+.2f}, {r['diff_hi']:+.2f}], {r['storms']} storms",
            transform=ax.get_yaxis_transform(), ha="left", va="center", fontsize=9, fontweight="bold", color=INK)
    ypos -= 1
    for j in (0, 1):
        v, lo, hi = r["adj"][j], r["adj_lo"][j], r["adj_hi"][j]
        if np.isfinite(v):
            ax.errorbar(v, ypos, xerr=[[v - lo], [hi - v]], fmt="o", color=C_DARK if j else C_LIGHT, ms=6, elinewidth=1.6, capsize=0)
        ax.plot(r["pooled"][j], ypos, marker="|", ms=12, mew=1.6, color=MUTED, ls="none")
        ticks.append(ypos)
        labs.append(f"{r['names'][j]}\n{int(r['county_weeks'][j])} county-weeks, {r['counties'][j]} counties")
        ypos -= 1
    ypos -= 0.4
ax.set_yticks(ticks)
ax.set_yticklabels(labs, fontsize=8.5)
ax.set_ylim(ypos - 0.5, 0.7)                                                # a spare row at the bottom for the legend
ax.axvline(0, color=FAINT, lw=1)
ax.axvline(1, color=FAINT, lw=1, ls="--")
ax.grid(axis="y", visible=False)
ax.set_xlabel("Displacement share: extra visits elsewhere ÷ visits lost at home (1 = all of it moved)")
ax.set_title("Q2 Who moved their activity (storm-adjusted, 95% intervals)")
ax.legend(handles=[Line2D([], [], marker="o", color=C_DARK, ls="none", label="storm-adjusted"),
                   Line2D([], [], marker="|", ms=12, mew=1.6, color=MUTED, ls="none", label="plain sums")], fontsize=8.5, loc="lower left", ncol=2)

ys = np.arange(len(MISS))[::-1]
for yi, (c, r) in zip(ys, MISS.iterrows()):
    if pd.notna(r["coef"]):
        step4 = c == "size_g"
        ax_color = C_GREY if step4 else C_DARK
        bx.errorbar(r["coef"], yi, xerr=[[r["coef"] - r["lo"]], [r["hi"] - r["coef"]]], fmt="o", color=ax_color, ms=6, elinewidth=1.6, capsize=0)
        bx.annotate(f"{r['coef']:+.1f}", (r["coef"], yi), xytext=(0, 6), textcoords="offset points", ha="center", va="bottom", fontsize=8.5, color=INK)
bx.set_yticks(ys)
bx.set_yticklabels([f"{r['label']}\n{r['counties_group']} vs {r['counties_reference']} counties, bootstrap p = {r['p_boot']:.3f}" for _, r in MISS.iterrows()], fontsize=8.5)
bx.set_ylim(-0.6, len(MISS) - 0.4)
bx.axvline(0, color=FAINT, lw=1)
bx.grid(axis="y", visible=False)
bx.set_xlabel("Residual difference within the same storm, log points\n(below 0: losses of the first group underestimated more)")
bx.set_title("Q3 Whose losses the model missed (95% intervals)")
fig.tight_layout()
fig.savefig(f"{OUT}/step5_fig2_who.png"); plt.show()

In [ ]:
# 12. Checklist and conclusions
def verdict(passed):
    return "pending" if passed is None else ("pass" if passed else "fail")

v = LED.loc["visits_in_chg", "coef"]
S2C = f"{OUT}/step2c_robustness.csv"
ref2c = np.nan
if os.path.exists(S2C):
    s = pd.read_csv(S2C)
    s = s[(s["sample"] == "all counties") & (s["controls"] == "HxW")]
    ref2c = float(s["coef"].iloc[0]) if len(s) else np.nan
EQ4 = f"{OUT}/step4_equity.csv"
ref4 = np.nan
if os.path.exists(EQ4):
    e = pd.read_csv(EQ4)
    e = e[e["sample"] == "7 storms (held out)"]
    ref4 = float(e["gap_rural_minus_other"].iloc[0]) if len(e) else np.nan
gap4 = MISS.loc["size_g", "raw_gap"]
hit_f = HITC.index
cov_inc, cov_rucc = float(ATTR.loc[hit_f, "income"].notna().mean()), float(ATTR.loc[hit_f, "rucc"].notna().mean())
n_emp = int(MAIN["emp_lp"].notna().sum())
few = [LABELS[y] for y in LED.index if not judged(LED.loc[y])]
storms_used = {s_: (Q2[s_]["storms"] if Q2.get(s_) is not None else 0) for s_ in SPLITS2}
checks = [
    ("Visits 72+ h coefficient matches Step 2c (difference)",
     "—" if np.isnan(ref2c) else f"{v - ref2c:+.3f} ({v:+.1f} vs {ref2c:+.1f})", "|·| < 0.05", None if np.isnan(ref2c) else bool(abs(v - ref2c) < 0.05)),
    ("Pooled rural residual gap matches Step 4 (difference)",
     "—" if np.isnan(ref4) else f"{gap4 - ref4:+.2e}", "|·| < 1e-6", None if np.isnan(ref4) else bool(abs(gap4 - ref4) < 1e-6)),
    ("Share of counties with 72+ h that have income data", round(cov_inc, 3), "≥ 0.95", bool(cov_inc >= 0.95)),
    ("Share of counties with 72+ h that have a rural-urban code", round(cov_rucc, 3), "≥ 0.95", bool(cov_rucc >= 0.95)),
    ("County-weeks in the main sample with Paychex employment", n_emp, "≥ 1,000", n_emp >= 1000),
    ("Outcomes with too few counties to judge", "; ".join(few) if few else "—", "none", not few),
    ("Storms in the Q2 comparison (income / metro)", f"{storms_used['income']} / {storms_used['metro']}", "≥ 2",
     bool(min(storms_used.values()) >= 2)),
]
check_tab = pd.DataFrame(checks, columns=["Indicator", "Value", "Threshold", "passed"])
check_tab["Verdict"] = check_tab.pop("passed").map(verdict)
check_tab.to_csv(f"{OUT}/step5_checklist.csv", index=False)
display(check_tab)

def fmt(r):
    return "—" if r is None or pd.isna(r["coef"]) else f"{r['coef']:+.1f} [{r['lo']:+.1f}, {r['hi']:+.1f}], p = {r['p_boot']:.3f}, {int(r['counties_72plus'])} counties"

def fmt2(r):
    return "—" if r is None else f"{r['diff']:+.2f} [{r['diff_lo']:+.2f}, {r['diff_hi']:+.2f}]; {r['names'][0]} {r['adj'][0]:.2f}, {r['names'][1]} {r['adj'][1]:.2f}"

def fmt3(r):
    return "—" if pd.isna(r["coef"]) else f"{r['coef']:+.1f} [{r['lo']:+.1f}, {r['hi']:+.1f}], p = {r['p_boot']:.3f}"

yes_no = lambda x: "too few counties to judge" if x is None else ("yes" if x else "no")
conclusions = pd.DataFrame([
    ("Q1a", f"revenue {fmt(R['revenue_lp'])}; businesses open {fmt(R['merchants_lp'])}", yes_no(Q1A)),
    ("Q1b", f"below − above median wage {fmt(R['emp_gap'])}", Q1B),
    ("Q1c", f"residents' spending {fmt(R['spend_lp'])}", Q1C),
    ("Q1d", f"revenue − visits {fmt(R['rev_minus_visits'])}", Q1D),
    ("Q2 income", fmt2(Q2.get("income")), Q2_VERDICT.get("income", "—")),
    ("Q2 metro", fmt2(Q2.get("metro")), Q2_VERDICT.get("metro", "—")),
    ("Q3 metro", fmt3(MISS.loc["metro_g"]), Q3_VERDICT["metro_g"]),
    ("Q3 income", fmt3(MISS.loc["income_g"]), Q3_VERDICT["income_g"]),
], columns=["Question", "Value", "Conclusion"])
conclusions.to_csv(f"{OUT}/step5_conclusions.csv", index=False)
conclusions